In this notebook, we analyze the scope diversity using the U-score and the R-score reported by Yang, Zhao, Luo, and co-workers (Angew. Chem. Int. Ed. 2026, 65, e2455429).

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..','..','..')))
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from Code.benchmark import Benchmark
from Code.utils import obtain_full_covar_matrix
from sklearn.preprocessing import MinMaxScaler
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, Draw
from rdkit import DataStructs
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import colorsys


# functions for U- and R-scores
from ScopeMap_Scores.evaluate import main as calculate_u_and_r_scores


# Doyle colors
doyle_colors = ["#CE4C6F", "#1561C2", "#188F9D","#C4ADA2","#515798", "#CB7D85", "#A9A9A9"]
# extension of palette with lighter and darker versions
def adjust_lightness(color, factor=1.2):
    """
    Function to make colors lighter (factor > 1) or darker (factor < 1).
    """
    r, g, b = mcolors.to_rgb(color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = max(0, min(1, l * factor))
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return mcolors.to_hex((r, g, b))

lighter = [adjust_lightness(c, 1.2) for c in doyle_colors]
darker  = [adjust_lightness(c, 0.7) for c in doyle_colors]
all_colors = doyle_colors + darker[::-1] + lighter[::-1] 

# Save the categorical colormap
cat_cmap = ListedColormap(all_colors, name="Doyle_cat")
plt.colormaps.register(cat_cmap)

# Define and save a continuous colormap
colors = [doyle_colors[1],"#FFFFFFD1",doyle_colors[0]]
cont_cmap = LinearSegmentedColormap.from_list("Doyle_cont", colors)
plt.colormaps.register(cont_cmap)
wdir = Path(".")


# General plt parameters
plt.rcParams.update({
    "axes.titlesize": 20,        # Subplot title
    "axes.labelsize": 16,        # X and Y labels
    "figure.titlesize": 24,      # Suptitle
    "xtick.labelsize": 14,       # X tick labels
    "ytick.labelsize": 14,       # Y tick labels
    "legend.fontsize": 14,       # Legend text
    "legend.title_fontsize": 14, # Legend titles
    "font.family": "Helvetica"   # Font
    })

In [2]:
# define a couple things
objectives = ["yield"]
directory = "."
wdir = Path(directory)
datasets = ["high","medium","low"]
dfs_labelled = {dset: pd.read_csv(f"./../Amide_data/Datasets/amide_dset_dft_subs_{dset}-yielding.csv", 
                                  index_col=0,header=0) for dset in datasets}

### Recalculate the diversity scores with the U-Score and the R-Score

In [3]:
# go through the different datasets
u_results_dict = {}
r_results_dict = {}
for dset in datasets:

    print("----------------------------------------------------------------")
    print(f"Calculating U- and R-scores for dataset {dset}...")
    print("----------------------------------------------------------------")

    df_labelled = dfs_labelled[dset]

    # save the search space smiles in a file (required input for the scores)
    df_smiles = pd.DataFrame({"smiles":df_labelled.index})
    df_smiles["smiles"] = df_smiles["smiles"].apply(lambda x: Chem.MolToSmiles(Chem.MolFromSmiles(x), isomericSmiles=True))
    df_smiles.to_csv("ScopeMap_Scores/space_smiles.csv", index=False)

    # calculate the U- and R-scores for the different acquisition functions and pruning settings
    u_results = pd.DataFrame(np.nan, index=["Results"], columns=[])
    r_results = pd.DataFrame(np.nan, index=["Results"], columns=[])
    for acq in ["EI","Random","Greedy","Conv. selection","Explorative"]:
        if acq == "Random":
            acq_label = "random-selection"
        elif acq == "Conv. selection":
            acq_label = "human-like-acq"
        else:
            acq_label = acq.lower()
        for pruning in [True,False]:
            if pruning:
                pruning_label = "_with-pruning"
                pruning_flag = "with"
            else:
                pruning_label = "_no-pruning"
                pruning_flag = "without"
            if acq == "Conv. selection":
                pruning_label = ""
            u_vals = []
            r_vals = []
            for filename in os.listdir(f"./Results_Data/{dset}-dataset/{acq_label}{pruning_label}/raw_data"):
                # open file and get the names of the selected samples
                df_filename = pd.read_csv(f"./Results_Data/{dset}-dataset/{acq_label}{pruning_label}/raw_data/{filename}",index_col=0,header=0)
                df_filename["eval_samples"] = df_filename["eval_samples"].apply(lambda x: [y.strip("'") for y in x[1:-1].split(', ')])
                samples = []
                for _,row in df_filename.iterrows():
                    samples.extend(row["eval_samples"])
                samples = [Chem.MolToSmiles(Chem.MolFromSmiles(x), isomericSmiles=True) for x in samples]
                # save the selected samples in a file
                pd.DataFrame({"smiles": samples}).to_csv(f"ScopeMap_Scores/selected_smiles_{filename}", index=False)

                # calculate the scores
                u_val, r_val = calculate_u_and_r_scores([
                    "--substrate-file", "ScopeMap_Scores/space_smiles.csv",
                    "--experimental-file", f"ScopeMap_Scores/selected_smiles_{filename}",
                ])
                # clean up the temporary input file and also the generated fingerprint file
                os.remove(f"ScopeMap_Scores/selected_smiles_{filename}")
                os.remove(f"fp_spoc_morgan41024_Maccs_selected_smiles_{filename}")
                u_vals.append(u_val)
                r_vals.append(r_val)
            u_results.loc["Results", acq+pruning_label] = np.mean(u_vals)
            r_results.loc["Results", acq+pruning_label] = np.mean(r_vals)
    u_results.rename(columns={"EI_pruning":"ScopeBO"},inplace=True)
    r_results.rename(columns={"EI_pruning":"ScopeBO"},inplace=True)
    label_dict = {col: col for col in u_results.columns}
    for key,val in label_dict.items():
        if "_no-pruning" in val:
            label_dict[key] = val.split("_")[0]
        elif "pruning" in val:
            label_dict[key] = val.split("_")[0] + " (pruned)"
    u_results.rename(columns=label_dict,inplace=True)
    r_results.rename(columns=label_dict,inplace=True)
    u_results.sort_values(by="Results",inplace=True, axis=1)
    r_results.sort_values(by="Results",inplace=True, axis=1)

    # remove the finger print file for the search space smiles and also the search space smiles file
    os.remove("fp_spoc_morgan41024_Maccs_space_smiles.csv")
    os.remove("ScopeMap_Scores/space_smiles.csv")

    print(f"U-Scores for dataset {dset}:")
    display(u_results)
    print(f"R-Scores for dataset {dset}:")
    display(r_results)

    u_results_dict[dset] = u_results
    r_results_dict[dset] = r_results

----------------------------------------------------------------
Calculating U- and R-scores for dataset high...
----------------------------------------------------------------
Arguments: Namespace(substrate_file='ScopeMap_Scores/space_smiles.csv', substrate_fp_file=None, experimental_file='ScopeMap_Scores/selected_smiles_27balanced_b3_V13_s15.csv', experimental_fp_file=None, distance_metric='euclidean', k_neighbors=5, random_seed=42, random_runs=5, not_feature_columns=['smiles'])
Starting Sampling Quality Evaluation

【Part 1: CVT and Kennard-Stone Sampling Evaluation】
--------------------------------------------------------------------------------

1. Reading data files...
   - Experimental data: (27, 1)
   - Sampling size set to experimental data length: 27
   - Fingerprint file 'fp_spoc_morgan41024_Maccs_space_smiles.csv' not found, generating...
   - Generated fingerprint file: fp_spoc_morgan41024_Maccs_space_smiles.csv
   - Combined complete space: (522, 1192)

2. Performing CVT 

,Greedy,Random,Conv. selection,Greedy (pruned),Random (pruned),EI,EI (pruned),Explorative (pruned),Explorative
Results,38.180625,54.436563,58.998092,63.712181,65.787992,69.212045,71.727113,79.533132,79.73699


R-Scores for dataset high:


,Random (pruned),Random,EI (pruned),Explorative,Explorative (pruned),EI,Greedy (pruned),Conv. selection,Greedy
Results,60.772182,70.563053,90.001479,97.441489,97.524235,98.058828,101.492902,105.358348,123.464551


----------------------------------------------------------------
Calculating U- and R-scores for dataset medium...
----------------------------------------------------------------
Arguments: Namespace(substrate_file='ScopeMap_Scores/space_smiles.csv', substrate_fp_file=None, experimental_file='ScopeMap_Scores/selected_smiles_27balanced_b3_V13_s15.csv', experimental_fp_file=None, distance_metric='euclidean', k_neighbors=5, random_seed=42, random_runs=5, not_feature_columns=['smiles'])
Starting Sampling Quality Evaluation

【Part 1: CVT and Kennard-Stone Sampling Evaluation】
--------------------------------------------------------------------------------

1. Reading data files...
   - Experimental data: (27, 1)
   - Sampling size set to experimental data length: 27
   - Fingerprint file 'fp_spoc_morgan41024_Maccs_space_smiles.csv' not found, generating...
   - Generated fingerprint file: fp_spoc_morgan41024_Maccs_space_smiles.csv
   - Combined complete space: (522, 1192)

2. Performing CV

...

,Random (pruned),Random,EI (pruned),Explorative (pruned),EI,Explorative,Greedy (pruned),Conv. selection,Greedy
Results,60.772182,70.563053,90.119144,91.487498,93.099952,94.533837,97.400881,101.338689,126.376528


----------------------------------------------------------------
Calculating U- and R-scores for dataset low...
----------------------------------------------------------------
Arguments: Namespace(substrate_file='ScopeMap_Scores/space_smiles.csv', substrate_fp_file=None, experimental_file='ScopeMap_Scores/selected_smiles_27balanced_b3_V13_s15.csv', experimental_fp_file=None, distance_metric='euclidean', k_neighbors=5, random_seed=42, random_runs=5, not_feature_columns=['smiles'])
Starting Sampling Quality Evaluation

【Part 1: CVT and Kennard-Stone Sampling Evaluation】
--------------------------------------------------------------------------------

1. Reading data files...
   - Experimental data: (27, 1)
   - Sampling size set to experimental data length: 27
   - Fingerprint file 'fp_spoc_morgan41024_Maccs_space_smiles.csv' not found, generating...
   - Generated fingerprint file: fp_spoc_morgan41024_Maccs_space_smiles.csv
   - Combined complete space: (522, 1192)

2. Performing CVT s

,Greedy,Random,Conv. selection,Greedy (pruned),Random (pruned),EI,EI (pruned),Explorative,Explorative (pruned)
Results,36.547777,54.436563,56.855458,64.780085,65.787992,67.75175,72.786708,78.508294,78.987223


R-Scores for dataset low:


,Random (pruned),Random,Explorative,EI (pruned),Explorative (pruned),Greedy (pruned),Conv. selection,EI,Greedy
Results,60.772182,70.563053,90.740384,94.489896,95.017187,102.189131,106.860878,107.802227,129.158311


In [4]:
for dset in datasets:
    # combine u and r (plus ranks) into a single dataframe
    u_results_ranked = u_results_dict[dset].rank(axis=1, method='min', ascending=False).astype(int)
    r_results_ranked = r_results_dict[dset].rank(axis=1, method='min', ascending=False).astype(int)
    u_results_combined = pd.concat([u_results_dict[dset], u_results_ranked], axis=0)
    r_results_combined = pd.concat([r_results_dict[dset], r_results_ranked], axis=0)
    results_combined = pd.concat([u_results_combined, r_results_combined], axis=0)
    results_combined.index = ["U-Scores","U-Ranks","R-Scores","R-Ranks"]
    results_combined.applymap(lambda x: round(x, 3) if isinstance(x, float) else x)
    results_combined.to_csv(f"U_and_R_Scores_{dset}.csv", index=True, header=True)
    print(f"Combined results for dataset {dset}:")
    display(results_combined)

Combined results for dataset high:


,Greedy,Random,Conv. selection,Greedy (pruned),Random (pruned),EI,EI (pruned),Explorative (pruned),Explorative
U-Scores,38.180625,54.436563,58.998092,63.712181,65.787992,69.212045,71.727113,79.533132,79.736990
U-Ranks,9.000000,8.000000,7.000000,6.000000,5.000000,4.000000,3.000000,2.000000,1.000000
R-Scores,123.464551,70.563053,105.358348,101.492902,60.772182,98.058828,90.001479,97.524235,97.441489
R-Ranks,1.000000,8.000000,2.000000,3.000000,9.000000,4.000000,7.000000,5.000000,6.000000


Combined results for dataset medium:


,Greedy,Random,Conv. selection,Greedy (pruned),Random (pruned),EI,EI (pruned),Explorative,Explorative (pruned)
U-Scores,42.087511,54.436563,58.217947,64.845083,65.787992,70.288265,72.222939,79.328756,79.572441
U-Ranks,9.000000,8.000000,7.000000,6.000000,5.000000,4.000000,3.000000,2.000000,1.000000
R-Scores,126.376528,70.563053,101.338689,97.400881,60.772182,93.099952,90.119144,94.533837,91.487498
R-Ranks,1.000000,8.000000,2.000000,3.000000,9.000000,5.000000,7.000000,4.000000,6.000000


Combined results for dataset low:


,Greedy,Random,Conv. selection,Greedy (pruned),Random (pruned),EI,EI (pruned),Explorative,Explorative (pruned)
U-Scores,36.547777,54.436563,56.855458,64.780085,65.787992,67.751750,72.786708,78.508294,78.987223
U-Ranks,9.000000,8.000000,7.000000,6.000000,5.000000,4.000000,3.000000,2.000000,1.000000
R-Scores,129.158311,70.563053,106.860878,102.189131,60.772182,107.802227,94.489896,90.740384,95.017187
R-Ranks,1.000000,8.000000,3.000000,4.000000,9.000000,2.000000,6.000000,7.000000,5.000000


Note: EI(pruned) is ScopeBO

In [3]:
datasets = ["high","medium","low"]

for dset in datasets:
    df = pd.read_csv(f"U_and_R_Scores_{dset}.csv", index_col=0, header=0)
    df = df.rename(columns={"EI (pruned)": "ScopeBO"})
    order = [
        "Greedy",
        "Greedy (pruned)",
        "EI",
        "ScopeBO",
        "Random",
        "Random (pruned)",
        "Explorative",
        "Explorative (pruned)",
        "Conv. selection",
    ]
    df = df.T.reindex(order)
    print(dset)
    display(df)

high


,U-Scores,U-Ranks,R-Scores,R-Ranks
Greedy,38.180625,9.0,123.464551,1.0
Greedy (pruned),63.712181,6.0,101.492902,3.0
EI,69.212045,4.0,98.058828,4.0
ScopeBO,71.727113,3.0,90.001479,7.0
Random,54.436563,8.0,70.563053,8.0
Random (pruned),65.787992,5.0,60.772182,9.0
Explorative,79.736990,1.0,97.441489,6.0
Explorative (pruned),79.533132,2.0,97.524235,5.0
Conv. selection,58.998092,7.0,105.358348,2.0


medium


,U-Scores,U-Ranks,R-Scores,R-Ranks
Greedy,42.087511,9.0,126.376528,1.0
Greedy (pruned),64.845083,6.0,97.400881,3.0
EI,70.288265,4.0,93.099952,5.0
ScopeBO,72.222939,3.0,90.119144,7.0
Random,54.436563,8.0,70.563053,8.0
Random (pruned),65.787992,5.0,60.772182,9.0
Explorative,79.328756,2.0,94.533837,4.0
Explorative (pruned),79.572441,1.0,91.487498,6.0
Conv. selection,58.217947,7.0,101.338689,2.0


low


,U-Scores,U-Ranks,R-Scores,R-Ranks
Greedy,36.547777,9.0,129.158311,1.0
Greedy (pruned),64.780085,6.0,102.189131,4.0
EI,67.751750,4.0,107.802227,2.0
ScopeBO,72.786708,3.0,94.489896,6.0
Random,54.436563,8.0,70.563053,8.0
Random (pruned),65.787992,5.0,60.772182,9.0
Explorative,78.508294,2.0,90.740384,7.0
Explorative (pruned),78.987223,1.0,95.017187,5.0
Conv. selection,56.855458,7.0,106.860878,3.0
